# ⚡ ClipStack — Free Cloud GPU Engine (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/openclaw-by-hiten/Clipstack/blob/main/ClipStack_Cloud_Engine.ipynb)
[![Live Web Studio](https://img.shields.io/badge/Live%20Studio-GitHub%20Pages-yellow?style=flat&logo=github)](https://openclaw-by-hiten.github.io/Clipstack/)

Run the entire ClipStack AI video clipping studio on Google's **free NVIDIA T4 GPU** and **100 GB cloud disk**.
- **0 MB** disk space used on your personal computer.
- **0%** local CPU/GPU load.
- Full AI video downloading, Whisper word-level transcription, and 9:16 vertical rendering in the cloud.

---

### 🚀 Quick Start (3 Steps):
1. **Runtime Check**: Click **Runtime** > **Change runtime type** in top menu and select **T4 GPU** (Free).
2. **Run All Cells**: Press **Runtime** > **Run all** (or `Ctrl + F9`).
3. **Connect to Web App**: Copy the generated `https://xxxx.trycloudflare.com` URL from Cell 3, open [https://openclaw-by-hiten.github.io/Clipstack/](https://openclaw-by-hiten.github.io/Clipstack/), click **Backend** in the top right, and paste it!


In [ ]:
# Step 1: Verify NVIDIA GPU & Install System Dependencies
!nvidia-smi

print('\n📦 Installing system dependencies (FFmpeg, audio codecs, DejaVu fonts)...')
!apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1 fonts-dejavu-core msttcorefonts > /dev/null 2>&1
print('✅ System dependencies installed successfully!')


In [ ]:
# Step 2: Clone or Update ClipStack Repository & Install Python Packages
import os

REPO_PATH = '/content/Clipstack'

if not os.path.exists(REPO_PATH):
    print('📥 Cloning ClipStack from GitHub...')
    !git clone https://github.com/openclaw-by-hiten/Clipstack.git $REPO_PATH
else:
    print('🔄 Pulling latest updates from GitHub...')
    !cd $REPO_PATH && git pull

%cd $REPO_PATH

print('\n📦 Installing Python requirements (FastAPI, Faster-Whisper, yt-dlp, etc.)...')
!pip install -q -r requirements.txt
print('✅ Python environment ready!')


In [ ]:
# Step 3: Start ClipStack FastAPI Server & Launch Cloudflare Tunnel
import os
import time
import subprocess
import urllib.request
import re
from IPython.display import display, HTML

# 1. Download official Cloudflare Tunnel binary for Linux amd64
cloudflared_path = '/usr/local/bin/cloudflared'
if not os.path.exists(cloudflared_path):
    print('⬇️ Downloading Cloudflare Tunnel (cloudflared)...')
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        cloudflared_path
    )
    os.chmod(cloudflared_path, 0o755)
    print('✅ cloudflared installed successfully!')

# 2. Terminate any previous instances
!pkill -f uvicorn > /dev/null 2>&1
!pkill -f cloudflared > /dev/null 2>&1
time.sleep(1)

# 3. Launch ClipStack FastAPI backend server
print('🚀 Launching ClipStack FastAPI Backend on port 8000...')
server_log = open('/content/server.log', 'w')
server_proc = subprocess.Popen(
    ['python3', '-m', 'uvicorn', 'server:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=server_log,
    stderr=subprocess.STDOUT,
    cwd='/content/Clipstack'
)
time.sleep(4)

# 4. Start Cloudflare Tunnel pointing to port 8000
print('🌐 Creating secure HTTPS tunnel via Cloudflare...')
tunnel_log = open('/content/tunnel.log', 'w')
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# 5. Extract public trycloudflare.com URL
public_url = None
start_time = time.time()
while time.time() - start_time < 35:
    line = tunnel_proc.stdout.readline()
    if line:
        tunnel_log.write(line)
        tunnel_log.flush()
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            public_url = match.group(0)
            break

if public_url:
    print('\n' + '=' * 70)
    print(f'🎉 CLIPSTACK CLOUD GPU ENGINE IS ONLINE!')
    print(f'👉 Option 1 (Direct Studio - Zero Setup): {public_url}')
    print(f'👉 Option 2 (GitHub Pages Live App): https://openclaw-by-hiten.github.io/Clipstack/?backend={public_url}')
    print('=' * 70 + '\n')
    display(HTML(f'''
    <div style="background: linear-gradient(135deg, #090d16, #111827); border: 2px solid #38bdf8; border-radius: 16px; padding: 24px; font-family: -apple-system, BlinkMacSystemFont, Segoe UI, Roboto, sans-serif; color: #f8fafc; max-width: 680px; box-shadow: 0 20px 25px -5px rgba(0,0,0,0.5);">
        <div style="display: flex; align-items: center; gap: 12px; margin-bottom: 16px;">
            <span style="font-size: 28px;">🚀</span>
            <div>
                <h2 style="margin: 0; color: #38bdf8; font-size: 20px; font-weight: 800;">ClipStack Cloud GPU Engine is Ready!</h2>
                <p style="margin: 4px 0 0 0; color: #94a3b8; font-size: 13px;">Choose either option below to open your video studio:</p>
            </div>
        </div>

        <!-- Option 1: Direct 1-Click Studio -->
        <div style="background: rgba(16, 185, 129, 0.1); border: 1px solid rgba(16, 185, 129, 0.4); border-radius: 12px; padding: 16px; margin-bottom: 14px;">
            <div style="display: flex; align-items: center; justify-content: space-between; margin-bottom: 8px;">
                <span style="font-weight: 700; color: #34d399; font-size: 14px;">⭐ Option 1: Direct 1-Click Studio (Recommended)</span>
                <span style="background: #065f46; color: #a7f3d0; font-size: 10px; font-weight: 700; padding: 2px 8px; border-radius: 9999px;">ZERO SETUP</span>
            </div>
            <p style="margin: 0 0 10px 0; font-size: 12px; color: #cbd5e1; line-height: 1.4;">
                Runs directly off this Colab GPU instance. No copy-pasting required!
            </p>
            <a href="{public_url}" target="_blank" style="display: inline-block; background: #10b981; color: #022c22; font-weight: 800; font-size: 13px; padding: 8px 18px; border-radius: 8px; text-decoration: none;">
                👉 Launch Direct Studio ↗
            </a>
        </div>

        <!-- Option 2: Auto-Connect GitHub Pages -->
        <div style="background: rgba(56, 189, 248, 0.08); border: 1px solid rgba(56, 189, 248, 0.3); border-radius: 12px; padding: 16px;">
            <div style="display: flex; align-items: center; justify-content: space-between; margin-bottom: 8px;">
                <span style="font-weight: 700; color: #38bdf8; font-size: 14px;">🌐 Option 2: GitHub Pages (Auto-Connected)</span>
                <span style="background: #0c4a6e; color: #bae6fd; font-size: 10px; font-weight: 700; padding: 2px 8px; border-radius: 9999px;">1-CLICK SYNC</span>
            </div>
            <p style="margin: 0 0 10px 0; font-size: 12px; color: #cbd5e1; line-height: 1.4;">
                Opens official GitHub Pages with this tunnel automatically connected via query parameter.
            </p>
            <a href="https://openclaw-by-hiten.github.io/Clipstack/?backend={public_url}" target="_blank" style="display: inline-block; background: #0284c7; color: #f0f9ff; font-weight: 800; font-size: 13px; padding: 8px 18px; border-radius: 8px; text-decoration: none;">
                👉 Launch GitHub Pages App ↗
            </a>
        </div>

        <div style="margin-top: 14px; padding-top: 12px; border-top: 1px solid #1f2937; display: flex; align-items: center; justify-content: space-between; font-size: 11px; color: #64748b;">
            <span>Cloud Backend Endpoint:</span>
            <span style="font-family: monospace; color: #94a3b8;">{public_url}</span>
        </div>
    </div>
    '''))
else:
    print('⚠️ Could not automatically detect tunnel URL in 35s. Please check /content/tunnel.log for details.')

# Stream tunnel output to keep cell alive
try:
    for line in tunnel_proc.stdout:
        tunnel_log.write(line)
        tunnel_log.flush()
except KeyboardInterrupt:
    print('🛑 Stopping server and tunnel...')
    server_proc.terminate()
    tunnel_proc.terminate()
